# Teaching Monster — Colab Pro+ GPU Notebook

**Runtime**: GPU (A100 on Pro+), High-RAM  

> Before running: **Runtime → Change runtime type → GPU → A100**, enable High-RAM.

## 0 · GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU — change runtime to GPU first!"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU  : {gpu.name}")
print(f"VRAM : {gpu.total_memory / 1e9:.1f} GB")
print(f"torch: {torch.__version__}")

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Clone repository from GitHub

In [ ]:
import os
REPO_URL = "https://github.com/ragiokay/TeachingMonster-released-main.git"
REPO_DIR = "/content/repo"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!ls

## 3 · Install system packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg libreoffice poppler-utils
print("System packages ready.")

## 4 · Install Python dependencies

In [ ]:
!pip -q install -U pip
!pip -q install -r requirements-colab-gpu.txt
print("Python packages ready.")

## 5 · Configure Gemini API key

In [ ]:
import getpass
from pathlib import Path

key = getpass.getpass("Paste your GEMINI_API_KEY: ")
Path("config/.env").write_text(f"GEMINI_API_KEY={key}\n", encoding="utf-8")
print("config/.env written.")

## 6 · Verify Gemini connectivity

In [ ]:
import yaml
from src.config_schema import AppConfig
from src.gemini_client import GeminiClient

cfg = AppConfig(**yaml.safe_load(open("config/default.yaml", encoding="utf-8")))
client = GeminiClient(cfg.llm)
client.load()
print("Gemini says:", client.generate("Reply with OK only.")[:80])

---
# API Server Mode (Competition)

## How it works

The judge's platform sends:
```json
POST /v1/video/generate
{ "request_id": "job-001", "course_requirement": "...", "student_persona": "..." }
```

Your server generates the video, uploads it to **Google Drive** (so the link stays valid 48+ hours), and responds:
```json
{ "video_url": "https://drive.google.com/uc?export=download&confirm=t&id=...", "subtitle_url": "...", "supplementary_url": null }
```

**Run cells 7 → 12 in order.**

## 7 · Authenticate Google Drive API

This lets the server upload generated videos to Drive automatically.

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Google Drive API authenticated.")

## 8 · Create public output folder in Drive

Videos will be uploaded here. The folder is set to 'anyone with link can download'.

In [ ]:
import os
from googleapiclient.discovery import build
from google.auth import default

creds, _ = default()
drive_service = build("drive", "v3", credentials=creds, cache_discovery=False)

# Create the output folder
folder = drive_service.files().create(
    body={"name": "TeachingMonster-Competition-Videos",
          "mimeType": "application/vnd.google-apps.folder"},
    fields="id",
).execute()
DRIVE_FOLDER_ID = folder["id"]

# Make it publicly readable
drive_service.permissions().create(
    fileId=DRIVE_FOLDER_ID,
    body={"type": "anyone", "role": "reader"},
).execute()

os.environ["DRIVE_OUTPUT_FOLDER_ID"] = DRIVE_FOLDER_ID
print(f"Drive folder created: https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
print(f"Folder ID saved to env: DRIVE_OUTPUT_FOLDER_ID={DRIVE_FOLDER_ID}")

## 9 · Install ngrok

In [ ]:
!pip -q install pyngrok

import getpass
from pyngrok import conf

# Get token at: https://dashboard.ngrok.com/get-started/your-authtoken
ngrok_token = getpass.getpass("Paste your ngrok authtoken: ")
conf.get_default().auth_token = ngrok_token
print("ngrok ready.")

## 10 · Start API server + ngrok tunnel

**This cell will take 10–20 minutes on first run** (downloading ~35 GB of model weights).  
Watch Cell 11 logs. Wait until you see `Pipeline loaded successfully!` before giving the URL to judges.

In [ ]:
import subprocess, os
from pyngrok import ngrok

os.environ.pop("TTS_FAST_MODE", None)
os.environ.pop("CURSOR_FAST_MODE", None)

# Open public tunnel
tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url
os.environ["BASE_URL"] = PUBLIC_URL

# Start server
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "src.server:app",
     "--host", "0.0.0.0", "--port", "8000"],
    env={**os.environ},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("=" * 60)
print(f"ENDPOINT FOR JUDGES:")
print(f"  POST {PUBLIC_URL}/v1/video/generate")
print("=" * 60)
print("Waiting for models to load (watch Cell 11 for progress)...")
print("Videos will be uploaded to Google Drive — links valid 48+ hours.")

## 11 · Watch server logs

Run this cell and wait for `Pipeline loaded successfully!`.  
**Do not give judges the URL until you see that message.**

In [ ]:
import io
for line in io.TextIOWrapper(server_proc.stdout, encoding="utf-8"):
    print(line, end="")
    if "Pipeline loaded successfully" in line:
        print("\n>>> Server ready! You can now give the endpoint URL to judges. <<<")
        break

## 12 · Dry-run test (confirms everything works end-to-end)

In [ ]:
import requests as req

# Dry-run: no video generated, just checks connectivity
resp = req.post(
    f"{PUBLIC_URL}/v1/video/generate",
    json={"request_id": "dry-run-001",
          "course_requirement": "Test",
          "student_persona": "Test"},
    headers={"X-Dry-Run": "true"},
    timeout=10,
)
assert resp.status_code == 200, f"Unexpected status: {resp.status_code}\n{resp.text}"
print("Dry-run OK:", resp.json())
print()
print("Give this to the judges:")
print(f"  POST {PUBLIC_URL}/v1/video/generate")
print()
print("Health check:")
print(f"  GET  {PUBLIC_URL}/health")

---
### Troubleshooting

| Symptom | Fix |
|---|---|
| `CUDA out of memory` | Enable High-RAM runtime, rerun from Cell 0 |
| Server stuck on model download | Wait — ~35 GB download takes 15–25 min |
| ngrok 401 error | Re-run Cell 9 with correct authtoken |
| Drive upload fails | Re-run Cell 7 (re-authenticate) |
| 504 from judges (timeout) | Pipeline took >30 min — check GPU assignment |
| Session about to expire | ngrok URL dies, but Drive links already returned stay valid |

### If the session disconnects mid-competition
1. Reopen Colab from the same notebook
2. Rerun Cells 0–12 (models re-download, ~20 min)
3. You get a new ngrok URL — update judges with the new endpoint
4. **Previously returned Drive video_urls are still valid** — those files are safe in Drive